# Setup — MECTESIS en Vertex AI Workbench

Ejecutar cada celda en orden. Solo la primera vez, en una instancia nueva.

**Repositorio:** https://github.com/DavidGuzzi/MECTESIS  
**Tiempo estimado:** ~5 min (la descarga de Chronos-2 tarda ~30-60 s la primera vez)

## 1 — Verificar Python y clonar el repositorio

In [ ]:
import sys
print(f"Python {sys.version}")

# Clonar el repositorio (solo si no existe ya)
import os
if not os.path.isdir("MECTESIS"):
    !git clone https://github.com/DavidGuzzi/MECTESIS.git
else:
    print("Directorio MECTESIS ya existe — saltando clone")
    !git -C MECTESIS pull

%cd MECTESIS
!pwd

## 2 — Instalar dependencias

In [ ]:
# Instalar el paquete mectesis en modo editable + todas las dependencias
!pip install -e . --quiet

# Si el entorno no tiene torch con CUDA, instalar la versión CPU ligera:
# !pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet

## 3 — Verificar que mectesis se importa correctamente

In [ ]:
from mectesis.dgp import ARpDGP, MAqDGP, ARMApqDGP, RandomWalk
from mectesis.models import ARIMAModel, ChronosModel
from mectesis.simulation import MonteCarloEngine
from mectesis.metrics import BiasVarianceMSE
print("mectesis importado correctamente")

## 4 — Cargar Chronos-2

La primera vez descarga ~1.5 GB desde Hugging Face. Las siguientes se carga desde caché.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

chronos = ChronosModel(device=device)
print("Chronos-2 listo")

## 5 — Crear directorio de resultados

In [ ]:
from pathlib import Path
results_dir = Path("results/univariate_v3")
results_dir.mkdir(parents=True, exist_ok=True)
print(f"Directorio listo: {results_dir.resolve()}")

## 6 — Smoke test rápido (opcional)

Verifica el pipeline completo con R=5, T=50, H=6.

In [ ]:
import numpy as np
from mectesis.dgp import ARpDGP
from mectesis.models import ARIMAModel
from mectesis.simulation import MonteCarloEngine

dgp = ARpDGP(phis=[0.5], sigma=1.0, seed=42)
models = [ARIMAModel((1, 0, 0))]
engine = MonteCarloEngine(dgp, models, seed=42)
res = engine.run_monte_carlo(n_sim=5, T=50, horizon=6, dgp_params={}, verbose=True)
print("
RMSE h=1:", round(float(res["ARIMA(1, 0, 0)"]["rmse"].iloc[0]), 4))
print("Smoke test OK")

---
## Listo para ejecutar el notebook principal

Una vez completados los pasos anteriores, abrir y ejecutar:

```
notebooks/experimentos_univariados_v3_cloud.ipynb
```

Los resultados se guardan en `results/univariate_v3/` (CSV por experimento + archivo `.log`).

Para ejecutar todo sin supervisión: **Kernel → Restart & Run All**.